# Inspect Bangladesh-2022-Sylhet chips

Loads the chip index CSV produced by `scripts/07_tile_to_chips.py` and renders quad-panel plots inline (pre-event VV, post-event VV, post minus pre difference, ground-truth flood label). Useful for spot-checking individual chips after the pipeline runs.

Plotting is delegated to the shared `scripts/utils/viz.py` helper so that this notebook stays in sync with `scripts/08_quality_check.py`.

In [ ]:
from pathlib import Path
import sys
import csv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from utils.viz import render_chip_panel

FINAL_DIR = PROJECT_ROOT / 'data' / 'final'
INDEX_PATH = FINAL_DIR / 'chip_index.csv'
assert INDEX_PATH.exists(), f'Run scripts/07_tile_to_chips.py first; missing {INDEX_PATH}'

In [ ]:
with open(INDEX_PATH) as f:
    rows = list(csv.DictReader(f))
print(f'Loaded {len(rows)} chips')
rows[:3]

## Render the first 5 chips inline

In [ ]:
%matplotlib inline
for row in rows[:5]:
    pre = PROJECT_ROOT / row['pre_path']
    post = PROJECT_ROOT / row['post_path']
    label = PROJECT_ROOT / row['label_path']
    render_chip_panel(row['chip_id'], pre, post, label, output_path=None, show_inline=True)

## Render a specific chip by ID

Replace `target_id` with the chip ID you want to inspect. Useful when reviewing a chip flagged during QA.

In [ ]:
target_id = 'sylhet_2022_chip_000017'
match = next((r for r in rows if r['chip_id'] == target_id), None)
if match is None:
    print(f'No chip with id {target_id} in the index')
else:
    render_chip_panel(
        match['chip_id'],
        PROJECT_ROOT / match['pre_path'],
        PROJECT_ROOT / match['post_path'],
        PROJECT_ROOT / match['label_path'],
        output_path=None,
        show_inline=True,
    )

## Distribution of flood pixel fraction across chips

In [ ]:
import matplotlib.pyplot as plt
fractions = [float(r['flood_pixel_fraction']) for r in rows]
plt.figure(figsize=(8, 4))
plt.hist(fractions, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('Flood pixel fraction per chip')
plt.ylabel('Number of chips')
plt.title(f'Flood pixel fraction distribution (n={len(fractions)})')
plt.tight_layout()
plt.show()